In [ ]:
Design Conditional Random Fields (CRFs) for POS tagging.

In [2]:
import csv
from sklearn_crfsuite import CRF
from sklearn.model_selection import train_test_split
from sklearn_crfsuite.metrics import flat_f1_score

# Read Pos_training data
def read_data(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            words = row['sentence'].split()
            tags = row['tags'].split()
            data.append(list(zip(words, tags)))
    return data

# Simplified feature extraction
def extract_features(sentence):
    features = []
    for i, (word, _) in enumerate(sentence):
        word_features = {
            'word': word.lower(),
            'is_upper': word.isupper(),
            'length': len(word)
        }
        if i > 0:
            word_features['prev_word'] = sentence[i-1][0].lower()
        else:
            word_features['prev_word'] = '<START>'
            
        if i < len(sentence)-1:
            word_features['next_word'] = sentence[i+1][0].lower()
        else:
            word_features['next_word'] = '<END>'
            
        features.append(word_features)
    return features

# Extract labels
def extract_labels(sentence):
    return [tag for _, tag in sentence]

# Train and evaluate CRF model
def train_and_predict(train_csv, input_csv, output_csv):
    training_data = read_data(train_csv)
    
    X = [extract_features(s) for s in training_data]
    y = [extract_labels(s) for s in training_data]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    
    crf = CRF(algorithm='lbfgs', max_iterations=100)
    crf.fit(X_train, y_train)
    
    y_pred = crf.predict(X_test)
    print(f"Model F1-score: {flat_f1_score(y_test, y_pred, average='weighted'):.3f}")
    
    # Predict and save results
    with open(input_csv, 'r') as infile, open(output_csv, 'w', newline='') as outfile:
        reader = csv.reader(infile)
        writer = csv.writer(outfile)
        writer.writerow(['Sentence', 'Predicted_Tags'])
        
        for row in reader:
            sentence = [(word, '') for word in row[0].split()]
            features = extract_features(sentence)
            tags = crf.predict([features])[0]
            writer.writerow([' '.join(word for word, _ in sentence), ' '.join(tags)])

# Execution flow
if __name__ == "__main__":
    train_csv = 'pos_train.csv'
    input_csv = 'sentences.csv'
    output_csv = 'pos_predictions.csv'
    
    train_and_predict(train_csv, input_csv, output_csv)
    print(f"Predictions saved to {output_csv}")


Model F1-score: 0.448
Predictions saved to pos_predictions.csv
